# English → Telugu Agriculture Glossary Translation

**IndicTrans2 + Google Colab**

This notebook extracts English glossary terms from a PDF, translates them into Telugu, and saves a CSV file.

## 1. Runtime setup

Use a **fresh Colab runtime**. GPU is optional but recommended.

The IndicTrans2 model is gated on Hugging Face, so you must first accept access on the model page and authenticate with a Hugging Face token.

In [1]:
# Install compatible dependencies
!pip -q install --upgrade --no-cache-dir \
    "transformers==4.46.3" \
    "huggingface-hub>=0.24,<1.0" \
    sentencepiece \
    sacremoses \
    accelerate \
    IndicTransToolkit \
    pdfplumber \
    pandas

# Remove torchvision if a broken preinstalled build causes Pillow/torchvision import errors.
!pip -q uninstall -y torchvision || true
!pip -q install --upgrade --no-cache-dir pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 283.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 314.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 199.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 262.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 252.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 178.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 196.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 171.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 256.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Restart the runtime now

After the installation cell finishes: **Runtime → Restart session**, then continue from the next cell.

In [2]:
import re
import torch
import pandas as pd
import pdfplumber

from huggingface_hub import login
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


Torch: 2.11.0+cu128
CUDA available: True


## 2. Hugging Face authentication

Before running this cell:
1. Log in to Hugging Face.
2. Open `ai4bharat/indictrans2-en-indic-dist-200M` and accept the repository access conditions.
3. Create a **Read** token.
4. Paste the token when prompted below.


In [5]:
# Run this only once per fresh session.
login()


In [6]:
MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
SRC_LANG = 'eng_Latn'
TGT_LANG = 'tel_Telu'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print('Loading model...')
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float32
).to(DEVICE)
model.eval()

ip = IndicProcessor(inference=True)
print(f'Model loaded successfully on {DEVICE}!')


Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

Model loaded successfully on cuda!


## 3. Upload the English glossary PDF

In [7]:
from google.colab import files

uploaded = files.upload()
PDF_PATH = next(iter(uploaded.keys()))
print('Using:', PDF_PATH)


Saving glossary.pdf to glossary.pdf
Using: glossary.pdf


In [8]:
def extract_pdf_text(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ''
            if text.strip():
                pages.append(text)
    return '\n'.join(pages)

raw_text = extract_pdf_text(PDF_PATH)
print(raw_text[:2000])
print('\nCharacters extracted:', len(raw_text))


Agriculture Glossary (English)
Source glossary for English fi Telugu SLM-based translation pipeline
Irrigation
The controlled application of water to crops through artificial means such as canals, sprinklers, or drip
systems.
Crop Rotation
The practice of growing different types of crops in the same area across sequential seasons to
maintain soil fertility.
Fertilizer
A substance added to soil to supply nutrients such as nitrogen, phosphorus, and potassium needed
for plant growth.
Pesticide
A chemical or biological agent used to deter, incapacitate, kill, or otherwise discourage pests that
damage crops.
Soil Erosion
The gradual wearing away and loss of the topsoil layer caused by wind, water, or poor land
management.
Harvest
The process of gathering mature crops from the fields at the end of a growing season.
Yield
The quantity of agricultural produce obtained per unit area of land, often measured in kilograms or
tonnes per hectare.
Agroforestry
A land-use system that integrates trees 

## 4. Convert extracted text into glossary entries

This cell treats non-empty lines as glossary entries and removes obvious page-number-only lines. Adjust it if your PDF has a different structure.

In [9]:
def clean_entries(text):
    entries = []
    for line in text.splitlines():
        line = re.sub(r'\s+', ' ', line).strip()
        if not line:
            continue
        if re.fullmatch(r'\d+', line):
            continue
        entries.append(line)
    return entries

entries = clean_entries(raw_text)
print('Total extracted entries:', len(entries))
for i, entry in enumerate(entries[:20], 1):
    print(f'{i}. {entry}')


Total extracted entries: 34
1. Agriculture Glossary (English)
2. Source glossary for English fi Telugu SLM-based translation pipeline
3. Irrigation
4. The controlled application of water to crops through artificial means such as canals, sprinklers, or drip
5. systems.
6. Crop Rotation
7. The practice of growing different types of crops in the same area across sequential seasons to
8. maintain soil fertility.
9. Fertilizer
10. A substance added to soil to supply nutrients such as nitrogen, phosphorus, and potassium needed
11. for plant growth.
12. Pesticide
13. A chemical or biological agent used to deter, incapacitate, kill, or otherwise discourage pests that
14. damage crops.
15. Soil Erosion
16. The gradual wearing away and loss of the topsoil layer caused by wind, water, or poor land
17. management.
18. Harvest
19. The process of gathering mature crops from the fields at the end of a growing season.
20. Yield


## 5. Translation function

Translations are processed in batches to reduce memory usage.

In [10]:
def translate_batch(sentences, batch_size=8, max_length=256):
    translated = []
    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]

        processed = ip.preprocess_batch(
            batch,
            src_lang=SRC_LANG,
            tgt_lang=TGT_LANG,
        )

        inputs = tokenizer(
            processed,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt',
        ).to(DEVICE)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                use_cache=True,
                min_length=0,
                max_length=max_length,
                num_beams=4,
                num_return_sequences=1,
            )

        decoded = tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        decoded = ip.postprocess_batch(decoded, lang=TGT_LANG)
        translated.extend(decoded)

        print(f'Translated {min(start + batch_size, len(sentences))}/{len(sentences)}')

    return translated


In [11]:
# Quick test before translating the full glossary
test_text = ['Agriculture is important for food production.']
test_translation = translate_batch(test_text, batch_size=1)
print('English:', test_text[0])
print('Telugu :', test_translation[0])


Translated 1/1
English: Agriculture is important for food production.
Telugu : ఆహార ఉత్పత్తికి వ్యవసాయం చాలా ముఖ్యం.


## 6. Translate the complete glossary

In [12]:
telugu_entries = translate_batch(entries, batch_size=8)
assert len(entries) == len(telugu_entries)
print('Translation complete!')


Translated 8/34
Translated 16/34
Translated 24/34
Translated 32/34
Translated 34/34
Translation complete!


## 7. Save results as CSV

In [13]:
df = pd.DataFrame({
    'English': entries,
    'Telugu': telugu_entries,
})

OUTPUT_FILE = 'telugu_glossary.csv'
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

display(df.head(20))
print('Saved:', OUTPUT_FILE)


,English,Telugu
0,Agriculture Glossary (English),వ్యవసాయ పదకోశం (ఆంగ్లం)
1,Source glossary for English fi Telugu SLM-base...,తెలుగు తెలుగు ఎస్ఎల్ఎం ఆధారిత అనువాద పైప్లైన్ ...
2,Irrigation,నీటిపారుదల
3,The controlled application of water to crops t...,"కాలువలు, స్ప్రింక్లర్లు లేదా బిందు వంటి కృత్రి..."
4,systems.,వ్యవస్థలు.
5,Crop Rotation,పంట రొటేషన్
6,The practice of growing different types of cro...,ఒకే ప్రాంతంలో వరుస సీజన్లలో వివిధ రకాల పంటలను ...
7,maintain soil fertility.,నేల యొక్క సంతానోత్పత్తిని నిర్వహించండి.
8,Fertilizer,ఎరువులు
9,A substance added to soil to supply nutrients ...,"అవసరమైన నత్రజని, భాస్వరం మరియు పొటాషియం వంటి ప..."


Saved: telugu_glossary.csv


In [14]:
from google.colab import files
files.download(OUTPUT_FILE)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>